# 🎧 StudyVoice AI — Convert Study Notes to MP3

StudyVoice AI lets you upload study notes and convert their text into spoken audio using the Sarvam AI Text-to-Speech API.

**Supported input files:** PDF, DOCX, TXT  
**Output:** MP3 audio sections packaged into a ZIP file

## How to use this notebook
1. Run the setup cell.
2. Configure your Sarvam API key securely in Colab Secrets (recommended) or enter it when prompted.
3. Upload your notes.
4. Choose the language for speech generation.
5. Generate and download the MP3 files.

**Important:** Selecting a language does not translate your notes. For best results, choose the language that matches the notes' text. Text-based PDFs are supported; scanned/image-only PDFs need OCR first.

## 1. Install dependencies

Run this cell first.

In [1]:
%pip -q install -U sarvamai pypdf python-docx
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.7/504.7 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 27.0 MB/s eta 0:00:00
Dependencies installed.


## 2. Connect to Sarvam AI securely

Recommended: in Google Colab, open the **Secrets** panel (key icon), add a secret named `SARVAM_API_KEY`, enter your API key as its value, and enable notebook access.

If the secret is not configured, this cell asks for the key using hidden input. Do not hard-code your key or commit it to GitHub.

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("SARVAM_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Enter your Sarvam API key (input is hidden): ").strip()

if not api_key:
    raise ValueError("No API key provided. Configure SARVAM_API_KEY and run this cell again.")

from sarvamai import SarvamAI
client = SarvamAI(api_subscription_key=api_key)
print("Sarvam AI client initialized.")

Sarvam AI client initialized.


## 3. Upload your notes

Run this cell and choose one PDF, DOCX, or TXT file from your computer.

In [3]:
from pathlib import Path
from io import BytesIO
from google.colab import files
from pypdf import PdfReader
from docx import Document

uploaded = files.upload()
if not uploaded:
    raise ValueError("No file was uploaded. Run this cell again and choose a notes file.")

file_name = next(iter(uploaded))
file_bytes = uploaded[file_name]
extension = Path(file_name).suffix.lower()

if extension == ".txt":
    notes_text = file_bytes.decode("utf-8", errors="replace")
elif extension == ".pdf":
    pdf = PdfReader(BytesIO(file_bytes))
    notes_text = "\n\n".join(page.extract_text() or "" for page in pdf.pages)
elif extension == ".docx":
    document = Document(BytesIO(file_bytes))
    notes_text = "\n\n".join(p.text for p in document.paragraphs)
else:
    raise ValueError("Unsupported file type. Please upload a .pdf, .docx, or .txt file.")

notes_text = notes_text.strip()
if not notes_text:
    raise ValueError(
        "No readable text was extracted. If this is a scanned/image-only PDF, OCR is needed first."
    )

print(f"Uploaded: {file_name}")
print(f"Extracted {len(notes_text):,} characters.")
print("\nPreview of extracted text:\n")
print(notes_text[:1000])

Saving Document 5.pdf to Document 5.pdf
Uploaded: Document 5.pdf
Extracted 3,860 characters.

Preview of extracted text:

Give 500 words funny story 
😂 The Day My Grandma Became an Influencer 
Once upon a time, in a small town in India, there lived a 22-year-old boy named Rahul. 
Rahul had only one dream in life: to become famous on Instagram without doing anything 
productive. 
Every morning, he woke up at 10 AM, took 47 selfies, posted one with the caption “Rise and 
grind 💪🔥”, and went back to sleep. 
His mother was tired of him. 
“Rahul! When will you get a job?” she shouted one morning. 
“Amma, please understand. I am building my personal brand.” 
“Your personal brand is currently eating my personal money!” she replied. 
One day, Rahul decided to teach his 70-year-old grandmother, Lakshmi, how to use 
Instagram. 
“Grandma, look! You can post photos, make reels, and become famous.” 
Grandma adjusted her glasses. “Will they give me money?” 
“Not immediately.” 
“Then why are you wast

## 4. Choose the speech language

This setting chooses the voice language; it does not translate the document.

In [5]:
language_map = {
    "english": "en-IN",
    "telugu": "te-IN",
    "hindi": "hi-IN",
}

language_choice = input("Choose speech language (English, Telugu, or Hindi): ").strip().lower()
if language_choice not in language_map:
    raise ValueError("Please enter English, Telugu, or Hindi.")

language_code = language_map[language_choice]
print("Selected language:", language_code)

Choose speech language (English, Telugu, or Hindi): English
Selected language: en-IN


## 5. Generate MP3 audio and download it

Long notes are split into smaller text sections before being sent to the speech API. Each section is saved as an MP3 file, then all MP3 files are packaged into one ZIP archive.

API use may consume credits and is subject to your Sarvam account limits.

In [6]:
import base64
import re
import zipfile
from pathlib import Path
from google.colab import files

def split_text(text, max_chars=2200):
    # Prefer paragraph boundaries, then split extra-long paragraphs safely.
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    chunks = []
    current = ""

    for paragraph in paragraphs:
        while len(paragraph) > max_chars:
            if current:
                chunks.append(current)
                current = ""
            cut = paragraph.rfind(" ", 0, max_chars)
            if cut < 1:
                cut = max_chars
            chunks.append(paragraph[:cut].strip())
            paragraph = paragraph[cut:].strip()

        if not paragraph:
            continue

        candidate = f"{current}\n\n{paragraph}".strip() if current else paragraph
        if len(candidate) <= max_chars:
            current = candidate
        else:
            if current:
                chunks.append(current)
            current = paragraph

    if current:
        chunks.append(current)
    return chunks

if not notes_text:
    raise ValueError("No notes text found. Run the upload and extraction cell first.")

text_chunks = split_text(notes_text)
output_dir = Path("studyvoice_mp3")
output_dir.mkdir(parents=True, exist_ok=True)
mp3_paths = []

print(f"Generating {len(text_chunks)} MP3 section(s)...")

for index, chunk in enumerate(text_chunks, start=1):
    print(f"Generating section {index}/{len(text_chunks)} ({len(chunk)} characters)...")
    response = client.text_to_speech.convert(
        text=chunk,
        language_code=language_code,
        model="bulbul:v3",
        output_audio_codec="mp3",
    )

    if not getattr(response, "audios", None) or not response.audios[0]:
        raise RuntimeError(f"No audio was returned for section {index}.")

    audio_bytes = base64.b64decode(response.audios[0])
    mp3_path = output_dir / f"studyvoice_part_{index:03d}.mp3"
    mp3_path.write_bytes(audio_bytes)
    mp3_paths.append(mp3_path)
    print(f"Saved: {mp3_path.name}")

if not mp3_paths:
    raise RuntimeError("No MP3 files were generated.")

zip_path = Path("studyvoice_mp3_files.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for mp3_path in mp3_paths:
        archive.write(mp3_path, arcname=mp3_path.name)

print(f"Success! Created {len(mp3_paths)} MP3 file(s).")
print(f"ZIP archive: {zip_path}")
files.download(str(zip_path))

Generating 3 MP3 section(s)...
Generating section 1/3 (1194 characters)...
Saved: studyvoice_part_001.mp3
Generating section 2/3 (2141 characters)...
Saved: studyvoice_part_002.mp3
Generating section 3/3 (518 characters)...
Saved: studyvoice_part_003.mp3
Success! Created 3 MP3 file(s).
ZIP archive: studyvoice_mp3_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Troubleshooting

- **`SecretNotFoundError`:** Add `SARVAM_API_KEY` in Colab Secrets and enable notebook access, or re-run the connection cell and enter the key when prompted.
- **`NameError: notes_text is not defined`:** Run the notebook cells from the top in order, especially the upload/extraction cell.
- **No text extracted from PDF:** The PDF may contain scanned page images. OCR is required before text-to-speech.
- **API error:** Check that your Sarvam API key is valid, your account has available usage, and the selected language is supported by your account/API version.

## Privacy and security

Do not commit your API key, personal notes, generated audio, or ZIP output to a public repository.